# z624 - Target escalado por TN_promedio, a nivel producto (aislado)

## Que prueba este notebook
Toma EXACTAMENTE la arquitectura de `LGB07_WF` (tu mejor resultado, 0.251): `product_id`, walk-forward 3 folds, features de `FE609`, Optuna. Cambia UNA sola cosa: el target deja de ser `log1p(tn)` y pasa a ser el escalado de la consigna nueva (`tn(p+2)/TN_promedio(p)`), calculado a nivel PRODUCTO (no cliente-producto, para no repetir el problema de amplificacion que ya vimos con la granularidad cliente-producto).

Objective: `regression`/`rmse` (NO tweedie) -- se aisla tambien esa variable, para no mezclar dos hipotesis (escalado vs. distribucion) en un solo test.

Si esto mejora sobre 0.251, el escalado por promedio aporta algo real, independiente de la granularidad y de tweedie. Si no mejora, el escalado en si no es la palanca.

In [1]:
!pip install -q lightgbm pyarrow optuna

In [2]:
import os
import numpy as np
import polars as pl
import lightgbm as lgb
import optuna
import warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [3]:
PARAM = {
    'experimento': 'LGB14_TARGET_ESCALADO',
    'kaggle_competition': 'labo-iii-2026-ba',
    'base_path': '/home/ds/exp/FE609/',
    'archivo_features': 'tb_features_FE609.parquet',
    'apredecir_path': '/home/ds/datasets/product_id_apredecir201912.txt',
    'horizonte_meses': 2,
    'periodo_ultimo_dato': 201912,
    'semilla': 102103,
    'n_trials': 50
}

ruta = os.path.join('/home/ds/exp', PARAM['experimento'])
os.makedirs(ruta, exist_ok=True)
print(ruta)

/home/ds/exp/LGB14_TARGET_ESCALADO


## 1. Cargar features (identicas a LGB07_WF, sin agregar ni sacar nada)

In [4]:
def periodo_a_meses(periodo: int) -> int:
    return (periodo // 100) * 12 + (periodo % 100)

df = pl.read_parquet(os.path.join(PARAM['base_path'], PARAM['archivo_features']))
df = df.sort(["product_id", "periodo"])
H = PARAM['horizonte_meses']

## 2. TN_promedio a nivel PRODUCTO (expanding, mismo criterio que z617 pero sin customer_id)

In [5]:
EPS = 1e-6

df = df.with_columns([
    pl.col("tn").cum_sum().over("product_id").alias("_cumsum_tn"),
    pl.col("tn").cum_count().over("product_id").alias("_cumcount"),
])
df = df.with_columns(
    (pl.col("_cumsum_tn") / pl.col("_cumcount")).alias("TN_promedio")
)
df = df.drop(["_cumsum_tn", "_cumcount"])

## 3. Target: clase_original_escalada = tn(p+2) / TN_promedio(p) -- naturalmente >=0

In [6]:
df = df.with_columns(
    pl.col("tn").shift(-H).over("product_id").alias("tn_target")
)
df = df.with_columns(
    (pl.col("tn_target") / (pl.col("TN_promedio") + EPS)).alias("clase_original_escalada")
)
df = df.with_columns(
    (pl.col("periodo_m") + H).alias("periodo_target_m")
)

print("minimo clase_original_escalada (debe ser >=0):", df["clase_original_escalada"].min())

minimo clase_original_escalada (debe ser >=0): 0.0


## 4. Walk-forward: mismos 3 cortes que z609

In [7]:
df_valido = df.filter(pl.col("clase_original_escalada").is_not_null())

cortes_calendario = [
    (201906, 201907, 201908),
    (201908, 201909, 201910),
    (201910, 201911, 201912),
]

folds = []
for train_max, valid_min, valid_max in cortes_calendario:
    m_train_max = periodo_a_meses(train_max)
    m_valid_min = periodo_a_meses(valid_min)
    m_valid_max = periodo_a_meses(valid_max)

    train_f = df_valido.filter(pl.col("periodo_target_m") <= m_train_max)
    valid_f = df_valido.filter(
        (pl.col("periodo_target_m") >= m_valid_min) & (pl.col("periodo_target_m") <= m_valid_max)
    )
    folds.append((train_f, valid_f))
    print(f"corte train<={train_max} valid={valid_min}-{valid_max}: train={train_f.height} valid={valid_f.height}")

corte train<=201906 valid=201907-201908: train=23645 valid=1788
corte train<=201908 valid=201909-201910: train=25433 valid=1816
corte train<=201910 valid=201911-201912: train=27249 valid=1827


## 5. Preparar matrices por fold
Features IDENTICAS a las de `LGB07_WF` -- se excluyen solo columnas de identificacion/target/intermedias, igual criterio que siempre.

In [8]:
cols_excluir = {"tn", "tn_target", "tn_shift1", "periodo", "periodo_target_m", "nacimiento_m",
                "TN_promedio", "clase_original_escalada"}
features = [c for c in df.columns if c not in cols_excluir]
categoricas = [c for c in ["product_id", "cat1", "cat2", "cat3", "brand", "descripcion"] if c in features]

def a_pandas(tabla):
    pdf = tabla.select(features + ["clase_original_escalada", "TN_promedio"]).to_pandas()
    for c in categoricas:
        pdf[c] = pdf[c].astype("category")
    return pdf

datasets_por_fold = []
for train_f, valid_f in folds:
    train_pd = a_pandas(train_f)
    valid_pd = a_pandas(valid_f)

    X_train = train_pd[features]
    y_train = train_pd["clase_original_escalada"]
    X_valid = valid_pd[features]
    y_valid = valid_pd["clase_original_escalada"]

    dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=categoricas,
                          params={'feature_pre_filter': False})
    dvalid = lgb.Dataset(X_valid, label=y_valid, categorical_feature=categoricas, reference=dtrain,
                          params={'feature_pre_filter': False})
    datasets_por_fold.append((dtrain, dvalid))

## 6. Optuna sobre el promedio de los 3 folds (objective=regression, NO tweedie)

In [9]:
def objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'verbosity': -1,
        'seed': PARAM['semilla'],
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 255),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 200),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'max_depth': trial.suggest_int('max_depth', -1, 15),
    }

    scores = []
    for dtrain, dvalid in datasets_por_fold:
        modelo = lgb.train(
            params, dtrain, num_boost_round=2000,
            valid_sets=[dvalid], callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )
        scores.append(modelo.best_score['valid_0']['rmse'])

    return float(np.mean(scores))

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=PARAM['semilla']))
study.optimize(objective, n_trials=PARAM['n_trials'], show_progress_bar=True)

print("mejor rmse promedio (3 folds):", study.best_value)
print("mejores params:", study.best_params)

  0%|          | 0/50 [00:00<?, ?it/s]

mejor rmse promedio (3 folds): 0.8414648201680709
mejores params: {'learning_rate': 0.19471477365785195, 'num_leaves': 190, 'min_data_in_leaf': 5, 'feature_fraction': 0.651225606475727, 'bagging_fraction': 0.6685954870217738, 'bagging_freq': 6, 'lambda_l1': 5.21970916360817e-06, 'lambda_l2': 4.979181689984344e-07, 'max_depth': -1}


## 7. Reentrenar final con el ultimo corte (train&le;201910, valid 201911-201912)

In [10]:
mejores_params = dict(study.best_params)
mejores_params.update({'objective': 'regression', 'metric': 'rmse', 'verbosity': -1, 'seed': PARAM['semilla']})

dtrain_final, dvalid_final = datasets_por_fold[-1]

modelo_final = lgb.train(
    mejores_params, dtrain_final, num_boost_round=2000,
    valid_sets=[dtrain_final, dvalid_final], valid_names=['train', 'valid'],
    callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(period=100)]
)

print("mejor iteracion:", modelo_final.best_iteration)

Training until validation scores don't improve for 100 rounds
[100]	train's rmse: 0.753452	valid's rmse: 6.19463
Early stopping, best iteration is:
[1]	train's rmse: 33.246	valid's rmse: 1.17133
mejor iteracion: 1


## 8. Total Error Rate sobre validacion (antes del submit)

In [11]:
_, valid_f_final = folds[-1]
valid_pd_final = a_pandas(valid_f_final)

pred_valid_escalada = modelo_final.predict(valid_pd_final[features], num_iteration=modelo_final.best_iteration)
pred_valid_tn = np.clip(pred_valid_escalada * valid_pd_final["TN_promedio"].to_numpy(), 0, None)
real_valid_tn = valid_f_final["tn_target"].to_numpy()

total_error_rate = np.abs(pred_valid_tn - real_valid_tn).sum() / real_valid_tn.sum()
print("Total Error Rate (validacion 201911-201912):", total_error_rate)

Total Error Rate (validacion 201911-201912): 0.6409143887672195


## 9. Prediccion para 202002 y submit

In [12]:
futuro = df.filter(pl.col("periodo") == PARAM['periodo_ultimo_dato'])
futuro_pd = futuro.select(features).to_pandas()
for c in categoricas:
    futuro_pd[c] = futuro_pd[c].astype("category")

pred_escalada = modelo_final.predict(futuro_pd, num_iteration=modelo_final.best_iteration)
tn_promedio_futuro = futuro["TN_promedio"].to_numpy()
pred_tn = np.clip(pred_escalada * tn_promedio_futuro, 0, None)

resultado = futuro.select(["product_id"]).to_pandas()
resultado["tn"] = pred_tn

In [13]:
apredecir = pl.read_csv(PARAM['apredecir_path'], separator="\t").to_pandas()

submit = apredecir[["product_id"]].merge(resultado, on="product_id", how="left")
print("nulos en submit (deberian ser 0):", submit["tn"].isna().sum())
submit["tn"] = submit["tn"].fillna(0.0)

archivo_submit = os.path.join(ruta, f"{PARAM['experimento']}_submit.csv")
submit.to_csv(archivo_submit, index=False)
print(archivo_submit)
submit.head()

nulos en submit (deberian ser 0): 0
/home/ds/exp/LGB14_TARGET_ESCALADO/LGB14_TARGET_ESCALADO_submit.csv


,product_id,tn
0,20001,1788.523890
1,20002,1291.011857
2,20003,1137.062812
3,20004,859.016008
4,20005,823.951578


## 10. Submit a Kaggle

In [14]:
def kaggle_submit(competencia, archivo, mensaje):
    comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
    os.system(comando)

kaggle_submit(PARAM['kaggle_competition'], archivo_submit, f"{PARAM['experimento']} target escalado TN_promedio, aislado")

100%|██████████| 18.7k/18.7k [00:00<00:00, 50.8kB/s]


97 submissions remaining today.
Successfully submitted to Labo III, 2026 BA